[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C29_Frontier_Interp_Course/05_interp_safety/05_interp_safety.ipynb)

# 05 · 可解释性用于安全（白盒监控 / 后门检测 / model diffing）

目标：在**合成数据**上从零实现前沿 interp 的安全用例，并用 `assert` 验证：
**后门检测**（defection probe + 手写 AUC，发作前预警）、**欺骗检测**（内部真值 vs 外部陈述）、**特征监控**（阈值 + precision/recall）、**crosscoder model diffing**（定位微调新增的后门特征）、以及一个**对抗规避**实验——同一探针被适应性对手绕过后 AUC 从 ~1.0 坍塌回 ~0.5。

路线：手写 AUC → 后门检测 → 欺骗检测 → 特征监控 P/R → crosscoder diffing → 对抗规避 → ✏️ 练习 ×4 → 📖 答案 → 🧪 真实安全证据强度胶囊。

> 立意：**行为是模型选择给你看的，内部状态是它不得不用的**。白盒监控读后者；但它是纵深防御的一层，不是绝对保证（第 6 节实证）。

## 1 · 手写 AUC（Mann–Whitney）+ 合成激活生成器

白盒监控的核心评估指标是 **AUC**（随机抽一对正负样本，监控给正样本更高分的概率），只依赖分数排序、对阈值与类别不平衡稳健。

先从零实现它（秩统计量版），再建一个统一的合成激活生成器：背景内容 + 触发器信号 + 危险信号。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def unit(v):
    return v / (np.linalg.norm(v) + 1e-12)

def auc_score(scores, labels):
    '''Mann-Whitney U 形式的 AUC：U / (n_pos * n_neg)。labels in {0,1}。'''
    scores = np.asarray(scores, float); labels = np.asarray(labels, int)
    pos = scores[labels == 1]; neg = scores[labels == 0]
    if len(pos) == 0 or len(neg) == 0:
        return 0.5
    order = np.argsort(scores, kind='mergesort')        # 稳定排序
    ranks = np.empty(len(scores)); ranks[order] = np.arange(1, len(scores) + 1)
    R_pos = ranks[labels == 1].sum()
    U = R_pos - len(pos) * (len(pos) + 1) / 2
    return U / (len(pos) * len(neg))

# 对拍：与「枚举所有正负对」的定义一致
def auc_pairwise(scores, labels):
    pos = scores[labels == 1]; neg = scores[labels == 0]
    return np.mean([(pp > nn) + 0.5 * (pp == nn) for pp in pos for nn in neg])

s = rng.standard_normal(200); y = (rng.random(200) < 0.4).astype(int)
assert abs(auc_score(s, y) - auc_pairwise(s, y)) < 1e-9, '秩版与枚举版应一致'
assert abs(auc_score(y.astype(float), y) - 1.0) < 1e-9, '完美分数 AUC=1'
print('✅ 手写 AUC 正确（秩统计量 == 枚举正负对）')

In [ ]:
# 统一的合成激活生成器：d 维，背景内容 + (触发时)附加信号
d = 32
g = np.random.default_rng(1)
base_dirs = g.standard_normal((8, d)); base_dirs /= np.linalg.norm(base_dirs, axis=1, keepdims=True)
trigger_dir = unit(g.standard_normal(d))   # 「触发器出现」方向
defect_dir  = unit(g.standard_normal(d))   # 「将要倒戈/使坏」方向
truth_dir   = unit(g.standard_normal(d))   # 内部「真值」方向

def gen_backdoor(n, added, p=0.5, seed=0, ncontent=8, noise=0.03):
    '''生成激活：每样本随机背景内容；以概率 p 触发，触发时叠加 added 向量。
       返回 (X[n,d], will_defect[n] 0/1)。added 即「模型在将要使坏时内部多出的信号」。'''
    rr = np.random.default_rng(seed)
    content = rr.standard_normal((n, ncontent)) @ base_dirs
    trig = (rr.random(n) < p).astype(float)
    X = content + noise * rr.standard_normal((n, d)) + trig[:, None] * added
    return X, trig

added = 1.5 * trigger_dir + 1.2 * defect_dir   # 触发时内部信号 = 触发器 + 倒戈
X, wd = gen_backdoor(2000, added, seed=2)
print('X', X.shape, '| 触发(将倒戈)比例', wd.mean().round(2))
assert X.shape == (2000, d)
print('✅ 合成后门激活就绪：触发样本携带「将要使坏」的内部信号')

## 2 · 后门检测：defection probe，在发作前预警

MacDiarmid 2024 的思想：用 **difference-of-means** 在「将倒戈 vs 不会」的内部激活上构造一个 **defection probe**（倒戈探针）。

关键是它读的是**内部激活**——在模型真正输出有害内容**之前**就能报警。我们在 held-out 集上算 AUC。

In [ ]:
def diff_means_probe(X, y):
    '''difference-of-means 探针方向：正类均值 - 负类均值，单位化。'''
    return unit(X[y == 1].mean(0) - X[y == 0].mean(0))

# train/test 划分，避免「在训练集上评估」
X, wd = gen_backdoor(3000, added, seed=2)
Xtr, Xte = X[:1500], X[1500:]; ytr, yte = wd[:1500], wd[1500:]
probe = diff_means_probe(Xtr, ytr)
scores_te = Xte @ probe
auc_te = auc_score(scores_te, yte)
print(f'defection probe 测试集 AUC = {auc_te:.3f}  (在有害输出【之前】读内部激活)')
assert auc_te > 0.95, '后门信号应在内部高度可分'
# 与随机方向基线对比
rand_dir = unit(rng.standard_normal(d))
auc_rand = auc_score(Xte @ rand_dir, yte)
print(f'随机方向基线 AUC = {auc_rand:.3f} (应≈0.5)')
assert abs(auc_rand - 0.5) < 0.15
print('✅ 后门检测：简单探针在发作前以高 AUC 抓住「将要使坏」的内部状态')

## 3 · 欺骗检测：内部真值读数 vs 外部陈述

**欺骗 = 内部状态与外部输出不一致**。设模型内部有可读的「真值方向」。欺骗样本上模型**嘴上说真、心里知道假**，于是「探针读到的内部真值」与「模型实际陈述」**背离**。背离率就是测谎信号。

In [ ]:
rr = np.random.default_rng(3); nn = 1500
internal_truth = (rr.random(nn) < 0.5).astype(float)   # 模型内部认为真(1)/假(0)
deceptive = (rr.random(nn) < 0.4).astype(float)        # 这条是否在撒谎
stated = np.where(deceptive == 1, 1 - internal_truth, internal_truth)  # 撒谎=翻转陈述

content = rr.standard_normal((nn, 8)) @ base_dirs
# 激活沿真值方向编码内部信念（与是否撒谎无关——撒谎只改输出）
Xd = content + 0.03 * rr.standard_normal((nn, d)) + (2 * internal_truth - 1)[:, None] * 1.0 * truth_dir

# 用真值方向读内部信念（阈值取中位数）
internal_read = (Xd @ truth_dir > np.median(Xd @ truth_dir)).astype(float)
mismatch = (internal_read != stated).astype(float)     # 内部读数 vs 外部陈述 背离
mm_dec = mismatch[deceptive == 1].mean(); mm_hon = mismatch[deceptive == 0].mean()
print(f'内外背离率: 欺骗样本={mm_dec:.2f}  诚实样本={mm_hon:.2f}')
assert mm_dec > mm_hon + 0.3, '欺骗样本的内外背离应显著更高'
print('✅ 欺骗检测：内部真值读数与外部陈述背离 -> 可测的诚实性信号')

## 4 · 特征监控：阈值一个 SAE 特征，算 precision/recall

比方向探针更可解释的监控：用一个**单义 SAE 特征**。设某特征专门读触发器方向（`relu(x·trigger_dir − b)`），部署时阈值它的激活即可报警。我们算 **precision/recall**（监控的两个核心运维指标）。

In [ ]:
def precision_recall(pred, true):
    '''pred/true 为 0/1 数组。返回 (precision, recall)。'''
    pred = pred.astype(int); true = true.astype(int)
    tp = ((pred == 1) & (true == 1)).sum()
    fp = ((pred == 1) & (true == 0)).sum()
    fn = ((pred == 0) & (true == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall

Xf, trig = gen_backdoor(2000, 1.6 * trigger_dir, p=0.3, seed=7)
feat_act = np.maximum(Xf @ trigger_dir - 0.5, 0.0)     # 一个读触发器的 SAE 特征
pred = (feat_act > 0.3).astype(int)                    # 监控：特征激活越阈值即报警
prec, rec = precision_recall(pred, trig)
print(f'特征监控: precision={prec:.3f}  recall={rec:.3f}')
assert prec > 0.9 and rec > 0.9, '专用特征应给出高精度高召回的监控'
print('✅ 特征监控：单义特征 + 阈值 = 可解释、高 P/R 的白盒监控器')

## 5 · Crosscoder model diffing：定位微调新增的后门特征

两个「模型」共享大部分特征，但 model B（微调后）多出一个**后门特征**。crosscoder 用共享字典让特征跨模型对齐——
我们用简化版（**跨模型最近余弦匹配**）把 B 的每个特征匹配到 A，那个**找不到 A 对应物（匹配余弦最低）**的 B 特征就是被植入的后门。

In [ ]:
def max_cos_to(Fb, Fa):
    '''Fb 每行特征到 Fa 任一特征的最大 |cos|。低 = 在 Fa 里找不到对应物。'''
    A = Fa / np.linalg.norm(Fa, axis=1, keepdims=True)
    B = Fb / np.linalg.norm(Fb, axis=1, keepdims=True)
    return np.abs(B @ A.T).max(1)

rA = np.random.default_rng(20); n_shared = 12
F_shared = rA.standard_normal((n_shared, d)); F_shared /= np.linalg.norm(F_shared, axis=1, keepdims=True)
backdoor_feat = unit(rA.standard_normal(d))
F_A = F_shared.copy()                              # model A 的特征
F_B = np.vstack([F_shared, backdoor_feat[None, :]]) # model B = 共享 + 1 个后门特征

mc = max_cos_to(F_B, F_A)
print('B 各特征到 A 的最大|cos|:', np.round(mc, 2))
detected = int(np.argmin(mc))                      # 最对不上的 = 后门
print(f'定位到的 B-独有特征 idx = {detected}  (真后门 idx = {len(F_B)-1})')
assert detected == len(F_B) - 1, 'model diffing 应定位到被植入的后门特征'
assert mc[detected] < 0.6 and mc[:-1].min() > 0.95, '后门特征显著对不上，共享特征都对得上'
print('✅ model diffing：跨模型特征对齐精确定位微调新增的后门特征')

## 6 · 局限实证：对抗规避让监控 AUC 坍塌

**最重要的一课**：白盒监控不是绝对保证。一旦探针成为优化目标，适应性对手可以把「将要使坏」的信号**移出探针可见的方向**。

我们把触发时的内部信号**投影到与已部署探针正交**的子空间（攻击者「绕开」探针），同一个探针的 AUC 从 ~1.0 坍塌回 ~0.5。

In [ ]:
# 已部署的探针（第2节训好的 probe）。对手已知它，于是规避：
added_evasive = added - (added @ probe) * probe       # 把内部信号投影到 probe 的正交补
print(f'规避后信号在探针方向的分量 = {added_evasive @ probe:.4f} (≈0 = 探针看不见了)')

Xe, wde = gen_backdoor(2000, added_evasive, seed=9)
auc_evasion = auc_score(Xe @ probe, wde)             # 用【同一个】探针监控
print(f'对抗规避后 同一探针 AUC = {auc_evasion:.3f}  (标准情形是 ~1.0)')
assert abs(auc_evasion - 0.5) < 0.12, '规避后探针应几乎失效（≈随机）'
print('✅ 局限实证：静态监控会被适应性对手绕过 —— interp 是纵深防御的一层，非绝对保证')
print('   （注意：后门【仍在】，行为照样会发作；只是探针看不见了 = streetlight）')

---
## ✏️ 练习 1：监控的运行点（阈值扫描）

监控要选一个**工作阈值**权衡 precision/recall。实现 `sweep_threshold(scores, labels, thresholds)`：对每个阈值算 `(precision, recall)`，返回列表 `[(thr, p, r), ...]`。复用第 4 节的 `precision_recall`。

验证：阈值升高 → recall 单调不增（更严 → 漏报更多）。

In [ ]:
def sweep_threshold(scores, labels, thresholds):
    # TODO: 对每个 thr in thresholds：pred = (scores > thr)，算 precision_recall，
    #       收集 (thr, precision, recall)。返回列表。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Xf2, trig2 = gen_backdoor(2000, 1.6 * trigger_dir, p=0.3, seed=15)
sc = np.maximum(Xf2 @ trigger_dir - 0.5, 0.0)
rows = sweep_threshold(sc, trig2, [0.0, 0.2, 0.5, 1.0, 2.0])
print('(thr, P, R):', [(t, round(pp, 2), round(rr_, 2)) for t, pp, rr_ in rows])
recalls = [rr_ for _, _, rr_ in rows]
assert all(recalls[i] >= recalls[i+1] - 1e-9 for i in range(len(recalls)-1)), '阈值升高召回应不增'
assert rows[0][2] > 0.9, '最低阈值应高召回'
print('✅ 练习 1 通过：监控运行点的 P/R 权衡')

## ✏️ 练习 2：后门定位（model diffing）

推广第 5 节：model B 可能有**多个**新增特征。实现 `find_novel_features(F_B, F_A, thresh=0.7)`：返回 B 中所有「到 A 最大 |cos| < thresh」的特征下标（即 A 里找不到对应物的新增/后门特征）。复用 `max_cos_to`。

In [ ]:
def find_novel_features(F_B, F_A, thresh=0.7):
    # TODO: 用 max_cos_to(F_B, F_A) 得到每个 B 特征的最大|cos|；
    #       返回 np.where(... < thresh)[0]（B 中的新增特征下标数组）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rB = np.random.default_rng(31); ns = 10
Fsh = rB.standard_normal((ns, d)); Fsh /= np.linalg.norm(Fsh, axis=1, keepdims=True)
nov1 = unit(rB.standard_normal(d)); nov2 = unit(rB.standard_normal(d))
F_A2 = Fsh.copy()
F_B2 = np.vstack([Fsh, nov1[None, :], nov2[None, :]])   # 2 个新增
novel = find_novel_features(F_B2, F_A2, thresh=0.7)
print('定位到的新增特征下标:', list(novel), '(真新增: [10, 11])')
assert set(novel.tolist()) == {10, 11}, '应精确定位两个新增特征'
print('✅ 练习 2 通过：model diffing 定位多个新增/后门特征')

## ✏️ 练习 3：审计证据强度（标准 vs 对抗）

一份审计要同时报「标准情形」与「对抗情形」的探针 AUC。实现 `audit_auc(added_signal, probe, seed)`：用 `gen_backdoor` 生成数据、用给定 `probe` 算 AUC，返回 AUC。

验证：标准信号 AUC 高；把信号投影出探针方向后 AUC ≈ 0.5（证据在对抗下失效）。

In [ ]:
def audit_auc(added_signal, probe, seed=0, n=2000):
    # TODO: X, y = gen_backdoor(n, added_signal, seed=seed)
    #        返回 auc_score(X @ probe, y)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
au_std = audit_auc(added, probe, seed=21)                       # 标准
added_adv = added - (added @ probe) * probe                     # 对抗：投影出探针方向
au_adv = audit_auc(added_adv, probe, seed=22)                   # 对抗
print(f'审计 AUC: 标准={au_std:.3f}  对抗规避={au_adv:.3f}')
assert au_std > 0.95, '标准情形证据应强'
assert abs(au_adv - 0.5) < 0.12, '对抗情形证据应失效（这正是要在审计里报告的局限）'
print('✅ 练习 3 通过：审计必须同时报标准与对抗，后者揭示证据的边界')

## ✏️ 练习 4：局限分析——监控被绕过，但行为没变

关键认知：探针失效**不代表后门消失**——后门行为照常发作，只是探针看不见了（streetlight）。

实现 `behavior_fires(added_signal, seed)`：返回触发样本里「危险方向 `defect_dir` 上的平均激活」（代理「后门行为强度」）。验证：规避前后**行为强度几乎不变**（后门还在），但第 3 题已显示**探针 AUC 坍塌**（监控瞎了）。

In [ ]:
def behavior_fires(added_signal, seed=0, n=2000):
    # TODO: X, y = gen_backdoor(n, added_signal, seed=seed)
    #        触发样本(y==1)在 defect_dir 上的平均投影 = 后门行为强度
    #        返回 (X[y==1] @ defect_dir).mean()
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 行为保持型规避：把危险信号塞进 defect_dir 中【探针看不见】的分量，
# 并缩放使其在 defect_dir 上的投影(=行为强度)与原来一致 -> 探针瞎了，但后门照常发作
d_perp = unit(defect_dir - (defect_dir @ probe) * probe)   # defect_dir 里探针正交的部分
scale = 1.2 / (d_perp @ defect_dir)                        # 缩放以保持行为强度
added_keepbehavior = scale * d_perp
b_std = behavior_fires(added, seed=41)
b_adv = behavior_fires(added_keepbehavior, seed=42)
auc_adv = auc_score(gen_backdoor(2000, added_keepbehavior, seed=42)[0] @ probe,
                    gen_backdoor(2000, added_keepbehavior, seed=42)[1])
print(f'后门行为强度: 规避前={b_std:.2f}  规避后={b_adv:.2f}  | 此时探针 AUC={auc_adv:.2f}')
assert b_adv > 0.8 * b_std, '行为保持型规避：后门行为应基本不变'
assert abs(auc_adv - 0.5) < 0.12, '但探针已瞎（AUC≈0.5）'
print('✅ 练习 4 通过：监控被绕过 ≠ 后门消失 —— streetlight 的实证，别把「没检到」当「安全」')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def sweep_threshold(scores, labels, thresholds):
    out = []
    for thr in thresholds:
        pp, rr_ = precision_recall((scores > thr).astype(int), labels)
        out.append((thr, pp, rr_))
    return out

In [ ]:
# 练习 2 参考答案
def find_novel_features(F_B, F_A, thresh=0.7):
    mc = max_cos_to(F_B, F_A)
    return np.where(mc < thresh)[0]

In [ ]:
# 练习 3 参考答案
def audit_auc(added_signal, probe, seed=0, n=2000):
    X, y = gen_backdoor(n, added_signal, seed=seed)
    return auc_score(X @ probe, y)

In [ ]:
# 练习 4 参考答案
def behavior_fires(added_signal, seed=0, n=2000):
    X, y = gen_backdoor(n, added_signal, seed=seed)
    return float((X[y == 1] @ defect_dir).mean())

---
## 🧪 真实数据胶囊：机制证据的强度梯子

把本课的安全证据按**强度**排个序（对应讲解第 6 节的表）。真实审计/safety case 里，证据越靠下越接近「因果坐实」、越可信。用一个简单打分体会「为什么单有探针报警不够，要往因果爬」。

In [ ]:
# 机制证据强度梯子（strength: 主观可信度权重 1-5；causal: 是否因果证据）
EVIDENCE = [
    dict(kind='probe_reading',  desc='defection probe AUC=0.97',        strength=2, causal=False),
    dict(kind='feature_monitor',desc='专用特征 P/R 双高',                strength=3, causal=False),
    dict(kind='model_diffing',  desc='crosscoder 发现可疑新增特征',      strength=3, causal=False),
    dict(kind='causal_interv',  desc='消融该特征->后门行为消失',          strength=5, causal=True),
]
print(f"{'证据类型':<18}{'强度':>5}{'因果':>6}  说明")
for e in EVIDENCE:
    print(f"{e['kind']:<18}{e['strength']:>5}{('是' if e['causal'] else '否'):>6}  {e['desc']}")
best = max(EVIDENCE, key=lambda e: e['strength'])
print(f"\n最强证据: {best['kind']} (强度{best['strength']}, 因果={best['causal']})")
assert best['causal'], '最强的安全证据应当是【因果】干预，而非相关性读数'
print('✅ 结论：探针/特征/diffing 给线索，因果干预才坐实 -> safety case 要往因果爬')

**🧪 胶囊练习**：实现 `safety_case_score(evidence_list)`：一份 safety case 的可信度 = 所有证据 strength 之和，但**若没有任何一条 causal 证据，则封顶在 6 分**（再多相关性线索也不够坐实）。返回最终分数。

In [ ]:
def safety_case_score(evidence_list):
    # TODO: total = sum(strength)；若没有任何 e['causal']==True，则 total = min(total, 6)
    #       返回 total
    raise NotImplementedError

In [ ]:
# 自测
only_corr = [e for e in EVIDENCE if not e['causal']]        # 只有相关性证据
with_causal = EVIDENCE                                       # 含因果
s_corr = safety_case_score(only_corr)
s_caus = safety_case_score(with_causal)
print(f'只有相关性证据 -> {s_corr} 分(封顶6)；含因果证据 -> {s_caus} 分')
assert s_corr == 6, '纯相关性证据应被封顶在 6'
assert s_caus == sum(e['strength'] for e in EVIDENCE), '含因果则按总和'
assert s_caus > s_corr, '因果证据应让 safety case 更可信'
print('✅ 胶囊练习通过：没有因果证据，再多相关性线索也不足以坐实安全')

In [ ]:
# 📖 胶囊参考答案
def safety_case_score(evidence_list):
    total = sum(e['strength'] for e in evidence_list)
    if not any(e['causal'] for e in evidence_list):
        total = min(total, 6)
    return total

### 小结
- **白盒监控** = 读内部激活而非只看输出；一次内积、近乎零成本，能抓行为评测看不到的欺骗/后门/藏拙。
- **后门检测**：difference-of-means 的 **defection probe** 在发作**前**高 AUC 预警（MacDiarmid 2024）。
- **欺骗检测**：内部真值读数 vs 外部陈述 **背离** = 可测的诚实性信号。
- **特征监控**：阈值一个单义 SAE 特征 = 可解释、高 P/R 的监控器。
- **crosscoder model diffing**：共享字典让特征跨模型对齐，**定位微调新增的(后门)特征**（Lindsey 2024）。
- **三条局限（写进 safety case 前必读）**：相关性≠因果、streetlight(没检到≠不存在)、对抗鲁棒性(监控会被绕过——第6节 AUC 从 1.0 坍塌到 0.5，但后门仍在)。
- **强度梯子**：探针→特征→diffing→**因果干预**；safety case 要往因果爬。interp 是纵深防御的一层，提供增量信心，非绝对保证。

🎓 **恭喜——你已走完整门前沿机制可解释性课程**：从叠加(00)→SAE 拆特征(01)→看清特征(02)→连成电路(03)→操控特征(04)→用于安全(05)。下一步：用 TransformerLens / SAELens / Neuronpedia 把这些 numpy 验证过的机制搬到真模型上。